# Optimization Campaign (HITL)

**Prerequisites:** TermNorm backend at `http://127.0.0.1:8000` | Groq API key in `.env` | Restart kernel after first sync

**Workflow:** Setup → Data → Explore → Optimize → Results

In [1]:
%load_ext autoreload
%autoreload 2

import json
from _campaign_lib import *

svc = await init_services()
TASK_DESCRIPTION = load_task_description(
    r"C:\Users\dsacc\OfficeAddinApps\TermNorm-excel\backend-api\config\LCA_INPUT_PATTERNS.md"
)

Backend: http://127.0.0.1:8000


2026-03-18 16:28:55 INFO     [httpx] HTTP Request: GET http://127.0.0.1:8000/pipeline "HTTP/1.1 200 OK"
2026-03-18 16:28:55 INFO     [api.services.pipeline_discovery] Matched known pipeline 'termnorm'; using enriched schema
2026-03-18 16:28:55 INFO     [api.services.campaign.campaign_init] Pipeline schema loaded: termnorm vv1.1


Pipeline: termnorm (6 steps)
Experiment: production_historical (40 queries, 93 session terms)
Experiment : production_historical
Mappings   : 887 total, 812 with verified ground truth
Queries    : 40  |  Session terms: 93
Loaded task description: 3751 chars from LCA_INPUT_PATTERNS.md


In [ ]:
campaign_config = {
    "sample_size": 15,              # queries per eval step (service default: all)
    "exploration_rate": 0.5,             # PRIMARY KNOB: 0.0=conservative, 1.0=aggressive
    "improvement_areas": "profile schema quality, web search relevance",
    "exclude_steps": ["llm_ranking"],    # steps to skip (e.g. ["entity_profiling"])
    "pipeline_overrides": {},
    "optimization": {
        "patience": 2,                   # default: 3
        "max_rounds": None,              # default: 10 (None = unlimited, patience-only stop)
        "degradation_threshold": 0.4,    # fraction of degraded queries to trigger escalation (0 = disabled)
        "enable_l2": True,               # L2 refine_context on escalation
        "enable_l3": True,               # L3 modify_plan on L2 stall
        "l2_patience": None,             # default: 2 (None = unlimited L2 rounds)
        "l3_patience": None,             # default: 1 (None = unlimited L3 rounds)
    },
    "eval_llm": {
        # --- Groq (free tier, open-source models) ---
        "model": "openai/gpt-oss-120b",
        # "model": "moonshotai/kimi-k2-instruct-0905"
        "provider_url": "https://api.groq.com/openai/v1/chat/completions",
        # --- Anthropic (cost: opus >> sonnet >> haiku) ---
        # "model": "claude-opus-4-6",          # best quality
        # "model": "claude-sonnet-4-6",      # good balance
        # "model": "claude-haiku-4-5-20251001",  # cheapest
        # "provider_url": "https://api.anthropic.com",
        "max_tokens": 2000,              # response length budget
    },
    "grid_search": {
        "context": "A terminology normalization pipeline that matches raw material "
                    "descriptions to standardized database terms using entity profiling "
                    "and candidate ranking.",
        "grid_budget": 35,               # default: 0 (full grid)
        "sample_size": 6,     # default: 0 (all queries)
        "shared_queries": False,          # default: True
    },
}

In [3]:
#@title Pipeline snapshot & params
pipeline_config_full = await show_pipeline_snapshot(svc)
pipeline_params = configure_pipeline(svc, campaign_config)

2026-03-18 16:28:55 INFO     [httpx] HTTP Request: GET http://127.0.0.1:8000/pipeline "HTTP/1.1 200 OK"


  PIPELINE SNAPSHOT: TermNorm v1.1
  Nodes:   ['fuzzy_matching', 'web_search', 'entity_profiling', 'token_matching', 'llm_ranking', 'direct_prompt']
  Schemas: ['entity_profile/1', 'llm_ranking_output/1']
  Prompts: ['entity_profiling/1', 'llm_ranking/1']

{
  "name": "TermNorm",
  "version": "v1.1",
  "available_models": [
    "moonshotai/kimi-k2-instruct-0905",
    "meta-llama/llama-4-scout-17b-16e-instruct",
    "moonshotai/kimi-k2-instruct",
    "openai/gpt-oss-120b"
  ],
  "nodes": {
    "fuzzy_matching": {
      "type": "DeterministicFunction",
      "config": {
        "threshold": 70,
        "scorer": "WRatio",
        "limit": 5
      }
    },
    "web_search": {
      "type": "ExternalService",
      "config": {
        "max_sites": 7,
        "num_results": 20,
        "content_char_limit": 800,
        "url_fetch_multiplier": 2,
        "fallback_keywords_limit": 8,
        "query_prefix": "",
        "query_suffix": "",
        "brave_api_timeout": 10,
        "scrape_tim

In [4]:
#@title 2. Data — Load datasets
# Set EXCEL_PATH to load from BOM-example.xlsx; leave empty to use stored data
EXCEL_PATH = r"C:\Users\dsacc\Desktop\project-TermNorm\OneDrive_2025-07-02\Austausch Beispiele\Prozessnamen\BOM-example.xlsx"  # e.g. "../data/BOM-example.xlsx"
FORCE_RELOAD = False  # Set True to re-read Excel and overwrite stored datasets

train_data, svc["session_terms"] = prepare_datasets(
    svc["store"], svc["backend_id"],
    excel_path=EXCEL_PATH or None,
    force=FORCE_RELOAD,
)


  Train              : 984 queries
  Test (processes)   : 82 queries
  Test (material)    : 165 queries
  ------------------------------------------------
  Combined queries   : 820 (deduplicated)
  Session identifiers: 94 unique targets


In [5]:
#@title Prepare evaluation context
campaign_rounds = []
baseline_results = []

baseline_ps, eval_data, backend_status = await prepare_eval_context(
    svc, train_data,
)

RUN_BASELINE = False  # Set True to evaluate baseline before exploration
if RUN_BASELINE:
    campaign_rounds, baseline_results = await run_baseline_eval(
        baseline_ps, eval_data, campaign_config, svc,
    )

2026-03-18 16:28:56 INFO     [httpx] HTTP Request: GET http://127.0.0.1:8000/status "HTTP/1.1 200 OK"



BACKEND STATUS
  Session Active                 True
  Active Sessions                1
  Terms Loaded                   94
  Match Database Identifiers     110
  Match Database Aliases         699
  Experiments Count              4
  Mappings Count                 1126
  Pipeline Version               v1.1
  Llm Provider                   groq
  Llm Model                      moonshotai/kimi-k2-instruct-0905
  ------------------------------------------------
  Experiments                   
    0_production_realtime        0 mappings
    1_production_historical      887 mappings
    2_bom_materials              159 mappings
    3_bom_processing             80 mappings

Evaluation data: 984 queries


In [6]:
#@title Experiment dashboard
# Set to a short hex ID (e.g. '68e2c5') to resume a specific experiment.
# The system adds prefixes (cycle_, scan_, etc.) per data type.
# Set to None to auto-detect from current campaign_config + eval_data.
EXPERIMENT_ID = '68e2c53845c3' #None

# When EXPERIMENT_ID is set, load stored config → overrides notebook variables
if EXPERIMENT_ID:
    stored_cfg = load_experiment_config(svc["store"], svc["backend_id"], EXPERIMENT_ID)
    if stored_cfg:
        pp_override = apply_experiment_overrides(campaign_config, stored_cfg)
        if pp_override:
            pipeline_params = pp_override
        print(f"  Loaded config from experiment {EXPERIMENT_ID}")

show_experiment_dashboard(
    svc=svc, experiment_id=EXPERIMENT_ID,
    campaign_config=campaign_config, eval_data=eval_data,
    pipeline_params=locals().get("pipeline_params"),
    baseline_prompt_state=campaign_rounds[0]["prompt_state"].model_dump() if campaign_rounds else None,
)

  Loaded config from experiment 68e2c53845c3

  EXPERIMENT: cycle_68e2c53845c3
  Status: completed  |  Rounds: 2  |  Best: 20.0%  |  Base: 20.0%
  Updated: 2026-03-18 15:27

  Config (copy to campaign_config to resume):
    max_rounds: 3
    patience: 2
    n_variants: 5
    creativity: 0.7
    improvement_threshold: 0.01
    model: openai/gpt-oss-120b
    temperature: 0.0
    sample_size: 15
    seed: 42
    pipeline_params: {max_token_candidates=30, profiling_schema=..., profiling_temperature=0.3, query_prefix=what material is, steps=...}


Diff: current config vs cycle_68e2c53845c3
  (identical — will resume this campaign)
  → Config does NOT match — update campaign_config to resume



{'campaign_id': 'cycle_68e2c53845c3',
 'created_at': '2026-03-12T16:07:56.174046+00:00',
 'updated_at': '2026-03-18T15:27:26.448382+00:00',
 'status': 'completed',
 'n_trials': 2,
 'best_accuracy': 0.2,
 'best_trial_id': 'round_0',
 'baseline_accuracy': 0.2,
 'trials': [{'trial_id': 'round_1',
   'round': 1,
   'label': 'round_1',
   'prompt_state_id': '',
   'accuracy': 0.2,
   'hits': 0,
   'total': 0,
   'improved': False,
   'created_at': ''},
  {'trial_id': 'round_0',
   'round': 0,
   'label': 'baseline',
   'prompt_state_id': '',
   'accuracy': 0.2,
   'hits': 0,
   'total': 0,
   'improved': False,
   'created_at': ''}],
 'type': 'feedback_cycle',
 'config': {'max_rounds': 3,
  'patience': 2,
  'n_variants': 5,
  'creativity': 0.7,
  'improvement_threshold': 0.01,
  'model': 'openai/gpt-oss-120b',
  'provider': None,
  'backend_url': 'http://127.0.0.1:8000',
  'backend_id': 'termnorm-local',
  'project_root': 'C:\\Users\\dsacc\\Desktop\\PromptPotter\\prompt-potter-optimizer\\.p

## 3. Explore

Two exploration paths: **Smart Search** (scan advisor + sensitivity scan) or **Grid Search** (brute-force sweep). Use one or both.

In [7]:
#@title 3a. Smart Search — Browse variant library
# display_variant_library()
# Filter examples:
# display_variant_library(source="PromptWizard")
# display_variant_library(axes=["thinking_style", "persona"])

In [8]:
# preview_advisor_prompt()
preview_advisor_prompt(campaign_config, svc, task_description="TASK_DESCRIPTION", raw=True)

2026-03-18 16:28:56 INFO     [api.services.search.smart_search] filter_variant_library: dropped all prompt_fields (llm_ranking not active)


You are an expert prompt optimization advisor. Recommend which axes (parameters and prompt fields) to prioritize in a sensitivity scan.

## Constraints (apply strictly)
- Do NOT recommend *_model axes — place them in axes_to_skip.
- Response must fit within 1500 tokens. Be terse.

## Pipeline: TermNorm AI terminology normalization pipeline
Steps execute sequentially — each step's output feeds the next:
[
  {
    "name": "cache_lookup",
    "node_role": "cache",
    "short_circuit": true
  },
  {
    "name": "fuzzy_matching",
    "node_role": "candidate_source",
    "short_circuit": true
  },
  {
    "name": "web_search",
    "node_role": "enricher"
  },
  {
    "name": "entity_profiling",
    "node_role": "enricher"
  },
  {
    "name": "token_matching",
    "node_role": "candidate_source"
  }
]

## Task Context
TASK_DESCRIPTION
## Tunable Parameters (per step)
[
  {
    "name": "fuzzy_matching",
    "param_keys": [
      "fuzzy_scorer",
      "fuzzy_threshold"
    ]
  },
  {
    "name

In [9]:
#@title Scan advisor
advisory, scan_variants, schema_labels = await run_scan_advisor(
    campaign_config, svc,
    task_description=locals().get("TASK_DESCRIPTION", ""),
)

2026-03-18 16:28:56 INFO     [api.services.search.smart_search] filter_variant_library: dropped all prompt_fields (llm_ranking not active)


SCAN ADVISOR -- pipeline-aware sensitivity setup
  Pipeline: termnorm (v1.1)
  Steps: ['cache_lookup', 'fuzzy_matching', 'web_search', 'entity_profiling', 'token_matching', 'llm_ranking']
  Excluded: ['llm_ranking']
  Task context: # Domain Context: Life Cycle Assessment (LCA) Terminology

This document capture...
  Calling openai/gpt-oss-120b ...



2026-03-18 16:29:01 INFO     [httpx] HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-03-18 16:29:01 WARNING  [api.models.schema_mutation] Remove target '/notes' not found in properties; skipping


----------------------------------------------------------------------
PRIORITY AXES (ranked by importance)
----------------------------------------------------------------------
  1. [HIGH] fuzzy_threshold (pipeline_param) -- step: fuzzy_matching
     Directly controls match acceptance
     Values: ['70', '80', '90']
  2. [MEDIUM] fuzzy_scorer (pipeline_param) -- step: fuzzy_matching
     Different scorers affect similarity capture
     Values: ['ratio', 'partial_ratio', 'token_set_ratio']
  3. [HIGH] query_prefix (pipeline_param) -- step: web_search
     Shapes initial search context
     Values: ['', 'material', 'chemical']
  4. [HIGH] query_suffix (pipeline_param) -- step: web_search
     Adds domain constraints to queries
     Values: ['', 'LCA', 'ecoinvent']
  5. [HIGH] profiling_prompt (pipeline_param) -- step: entity_profiling
     Guides extraction focus
     Values: ['', 'Extract concise LCA‑relevant profile']
  6. [MEDIUM] profiling_schema (pipeline_param) -- step: entity_pr

In [10]:
#@title Scan variant config (edit suggested values or add your own)
# Schema axes: mutation tuples ("-", path), ("+", path, type, req, desc),
# ("~", old, new, type, req, desc). Non-schema axes: plain value lists.

scan_sample_size = 10  # queries per scan variant (0 = use all)

scan_variants = {
    'max_token_candidates': [10, 30, 50],
    'query_prefix': ['what material is', 'identify LCA database name for', 'translate trade name'],
    'profiling_schema': [
        [['+', 'geography_scope', 'array', False, 'Relevant ecoinvent/SimaPro geography codes inferred from context, e.g. GLO, RER, CH, RNA'], ['+', 'database_format_hint', 'string', False, "Best-guess ecoinvent-style name fragment for this entity, e.g. 'market for polyethylene, high density'"]],
        # [['-', 'manufacturing_processes'], ['-', 'applications'], ['+', 'lca_synonyms', 'array', False, 'Terms likely to appear verbatim in LCA database entry names for this entity'], ['+', 'no_match_signal', 'string', False, 'Brief reasoning on whether a database match is likely to exist or not']],
        # [['~', 'classification_aliases', 'lca_classification_aliases', 'array', False, 'Expert-level aliases specifically aligned with LCA database naming conventions, including ecoinvent activity names and SimaPro process names'], ['+', 'geography_scope', 'array', False, 'Relevant ecoinvent/SimaPro geography codes inferred from context, e.g. GLO, RER, CH, RNA']],
        [['+', 'lca_database_names', 'array', True, "Likely ecoinvent or GaBi database entry names that would match this entity, using standard LCA database naming conventions like 'market for X | X | cut-off, U'"]], 
        [['-', 'manufacturing_processes'], ['-', 'applications'], ['+', 'lca_database_names', 'array', True, 'Likely ecoinvent or GaBi database entry names for this entity using standard LCA naming conventions']],
        [['~', 'notes', 'material_category', 'string', True, "The broad LCA material category this entity belongs to, e.g. 'polyethylene', 'brass', 'steel'"]]
    ],
    'profiling_temperature': [0.0, 0.3, 0.7],
    # 'profiling_max_tokens': [512, 1024, 2048], # -> Going to cause lots of Errors.
    'raw_content_limit': [1000, 2500, 8000],
}
scan_variants, schema_labels = resolve_scan_variants(scan_variants, svc=svc)

  max_token_candidates: [10, 30, 50]
  query_prefix: ['what material is', 'identify LCA database name for', 'translate trade name']
  profiling_schema: (baseline + 4 mutations)
    [0] (baseline)
    [1] ('+', 'geography_scope', 'array', False, 'Relevant ecoinvent/SimaPro geography codes inferred from context, e.g. GLO, RER, CH, RNA'), ('+', 'database_format_hint', 'string', False, 'Best-guess ecoinvent-style name fragment for this entity, e.g. 'market for polyethylene, high density'')
    [2] ('+', 'lca_database_names', 'array', True, 'Likely ecoinvent or GaBi database entry names that would match this entity, using standard LCA database naming conventions like 'market for X | X | cut-off, U'')
    [3] ('-', 'manufacturing_processes'), ('-', 'applications'), ('+', 'lca_database_names', 'array', True, 'Likely ecoinvent or GaBi database entry names for this entity using standard LCA naming conventions')
    [4] ('~', 'notes', 'material_category', 'string', True, 'The broad LCA material 

In [11]:
#@title Prepare scan baseline
# Scan always uses fresh pipeline defaults (not experiment overrides) so the
# baseline content hash matches previous runs regardless of EXPERIMENT_ID.
scan_pipeline_params = configure_pipeline(svc, campaign_config)
scan_baseline_sp, scan_coverage = await prepare_scan_baseline(
    baseline_ps, campaign_config,
    pipeline_params=scan_pipeline_params,
    svc=svc, scan_variants=scan_variants,
)

2026-03-18 16:29:01 INFO     [api.services.search.context] restructure_context_cached: hit (alias group)


Active steps: ['cache_lookup', 'fuzzy_matching', 'web_search', 'entity_profiling', 'token_matching']
  Excluded: ['llm_ranking']
  Restructured baseline fields (cached):
    persona: You are a candidate evaluation expert.
    task_intent: Summarize the entity profile, identify its category and key distinguishing featu...
    problem_description: Given an entity_profile_json and a list of candidate matches, produce a concise ...
    instruction: TASK 1: Summarize the profile in 1‑2 sentences, identify entity_category, and li...
    thinking_style: Think step by step.
    answer_format: JSON with keys "reasoning" (string) and "ranked_candidates" (array of objects co...
  Search baseline: 785b82bca0d5 (render: 1154 chars)


2026-03-18 16:29:01 INFO     [api.services.search.coverage] build_prompt_result_index: 84 runs -> 15 unique prompts, 3027 total query results



  Historical data: 3027 results across 15 unique prompts
  Baseline alias group: 14889901 ↔ 239cefb8 ↔ 24aeb9e7 ↔ 2c14e76c ↔ 2d9d15f6 ↔ 41f88bae ↔ 44eb12a0 ↔ 4ff72b79 ↔ 61ad2b63 ↔ 82f3e7e2 ↔ 830faccd ↔ 9e4f0633 ↔ adb2589d ↔ aeb18154 ↔ bcdc7b72 ↔ c2a36fe9 ↔ c311136d ↔ cec84ce0 ↔ d018a6dc ↔ e169ed86 (20 prompts linked)
  Matching runs: 83, 3012 cached results

  Scan variant coverage (83 matching runs):
    max_token_candidates     10→4 ✓  30→12 ✓  50→4 ✓  (+8 other)
    query_prefix             what material is→12 ✓  identify LCA database name for→1 ✓  translate trade name→1 ✓  (+8 other)
    profiling_schema         {"properties": {"alternative_names": {"items": {"type": "string"}, "type": "array"}, "applications": {"description": "Direct and derived applications based on product characteristics", "items": {"type": "string"}, "type": "array"}, "classification_aliases": {"description": "Full spectrum of valid ways this entity could be referenced using expert-level terminology, from pre

In [12]:
#@title Sensitivity scan
scan_df, axis_profiles = await sensitivity_scan(
    scan_baseline_sp, scan_variants, eval_data,
    sample_size=scan_sample_size,
    svc=svc, experiment_id=EXPERIMENT_ID or "",
)

Running sensitivity scan...

  Baseline field values:
    persona: You are a candidate evaluation expert.
    task_intent: Summarize the entity profile, identify its category and key distinguishing featu...
    problem_description: Given an entity_profile_json and a list of candidate matches, produce a concise ...
    instruction: TASK 1: Summarize the profile in 1‑2 sentences, identify entity_category, and li...
    thinking_style: Think step by step.
    answer_format: JSON with keys "reasoning" (string) and "ranked_candidates" (array of objects co...

  Axes: 5, variants: 17, queries/variant: 10, cached results: 11490
  Estimated calls: ~170
  Evaluating baseline...
        MISS 2/20  [token] 📖  PA66-GF25 ULTRAMID A3UG5 RAL7035 grey          -> Glass fibre reinforced plastic | 95 4.0s
        HIT   [token] 📖  Stainless steel EN 10270-3/winding             -> Wire drawing, steel {RER}| wire dra 4.1s
        MISS 5/20  [token] 📖  SJRG0010-ABS/molding                           -> Acryl

In [13]:
# #@title Scan analytics (uncomment to display)
# if scan_df is not None and not scan_df.empty:
#     show_scan_leaderboard(scan_df, axis_profiles)
#     difficulty_df = show_scan_query_difficulty(svc["store"], svc["backend_id"])

In [14]:
#@title Select scan winner & seed campaign
best_sp = seed_campaign_from_scan(
    scan_df, axis_profiles, scan_baseline_sp, scan_variants,
    campaign_rounds, campaign_config,
)

2026-03-18 16:29:01 INFO     [api.services.search.scan_winner] select_scan_winner: 0 prompt changes, 4 param changes from 4 improving axes


Selected best from 4 improving axes:
  max_token_candidates      best_delta=+18.0%  value_idx=1  acc=30.0%
  query_prefix              best_delta=+9.2%  value_idx=0  acc=20.0%
  profiling_schema          best_delta=+9.2%  value_idx=2  acc=20.0%
  profiling_temperature     best_delta=+9.0%  value_idx=1  acc=20.0%
Pipeline params updated: {'steps': ['cache_lookup', 'fuzzy_matching', 'web_search', 'entity_profiling', 'token_matching'], 'max_token_candidates': 30, 'query_prefix': 'what material is', 'profiling_schema': {'type': 'object', 'properties': {'entity_name': {'type': 'string'}, 'core_concept': {'type': 'string', 'description': 'The single word that defines what this expression represents'}, 'distinguishing_features': {'type': 'array', 'items': {'type': 'string'}}, 'key_properties': {'type': 'array', 'items': {'type': 'string'}}, 'technical_specifications': {'type': 'array', 'items': {'type': 'string'}, 'description': 'Explicit technical specs, dimensions, codes, ratings, tolerance

### 3b. Grid Search

<details>
<summary>Grid search cells (click to expand)</summary>

Systematic sweep of the prompt configuration space. Maps the accuracy landscape before hill-climbing. All cells below are commented out by default.

**To activate:** uncomment cells below and run in order. Grid search evaluates all combinations of prompt fields and pipeline params — expect 100–500+ backend calls depending on `grid_budget` and `sample_size` in `campaign_config["grid_search"]`.

</details>

In [15]:
# #@title Grid campaign overview (existing plans)
# merge_plans = False  # Set True to combine results from multiple plans
# grid_overview = show_grid_overview(svc, campaign_config, merge_plans=merge_plans)
# merged_grid_df = grid_overview.get("merged_grid_df")

In [16]:
# #@title Build or resume grid plan
# gs = campaign_config["grid_search"]

# llm_client, llm_model = setup_llm(campaign_config)

# (
#     grid_plan_id, grid_points, grid_state_lookup,
#     grid_axes, layer1_fields, grid_baseline,
# ) = await resume_or_build_grid(
#     campaign_config, baseline, llm_client, llm_model,
#     svc["store"], svc["backend_id"],
#     improvement_areas=campaign_config.get("improvement_areas", ""),
# )

# print(f"Grid points: {len(grid_points)}")
# print(f"Plan ID: {grid_plan_id}")

In [17]:
# #@title Run grid search
# grid_df = await run_grid_search(
#     grid_points, grid_state_lookup, eval_data,
#     campaign_config["eval_llm"],
#     plan_id=grid_plan_id,
#     svc=svc,
#     pipeline_params=campaign_config.get("pipeline_params"),
#     sample_size=gs.get("sample_size", 1),
#     shared_queries=gs.get("shared_queries", False),
#     grid_seed=gs.get("seed", 42),
# )

In [18]:
# #@title Display grid results
# _display_df = merged_grid_df if merged_grid_df is not None else grid_df
# display_grid_results(_display_df, grid_axes, top_k=gs.get("top_k", 5))

In [19]:
# #@title LLM analysis of grid results
# _analysis_df = merged_grid_df if merged_grid_df is not None else grid_df
# llm_client, llm_model = setup_llm(campaign_config)
# grid_analysis = await analyze_grid_results(
#     _analysis_df, grid_axes, llm_client, model=llm_model,
# )

In [20]:
# #@title Select grid winner and seed campaign
# grid_winner = select_and_seed_grid_winner(
#     grid_df, merged_grid_df, grid_state_lookup,
#     grid_overview.get("plan_dfs", {}), svc, campaign_rounds,
# )

## 4. Optimize

Two modes: **Semi-automatic** (feedback cycle with patience-based auto-stop) or **Manual** (one round at a time).

In [21]:
#@title Feedback cycle preflight
scan_context = show_feedback_preflight(
    campaign_rounds, eval_data, campaign_config,
    pipeline_params=pipeline_params,
    scan_df=locals().get("scan_df"),
    axis_profiles=locals().get("axis_profiles"),
    scan_variants=locals().get("scan_variants"),
    difficulty_df=locals().get("difficulty_df"),
)


  FEEDBACK CYCLE PRE-FLIGHT
  Baseline accuracy      : 10.0%
  Baseline prompt        : TASK 1: Summarize the profile in 1‑2 sentences, identify entity_category, and li...
  ------------------------------------------------------------------
  Max rounds             : 3
  Candidates per round   : 5
  Queries per eval       : 15 of 984
  Improvement threshold  : 1.0%
  Patience (L1)          : 2 rounds
  L2 (refine context)    : disabled
  L3 (modify plan)       : disabled
  ------------------------------------------------------------------
  Candidate model        : openai/gpt-oss-120b
  Creativity             : 0.7
  Pipeline               : 5 of 6 steps
    Steps                : cache_lookup, fuzzy_matching, web_search, entity_profiling, token_matching
    Excluded             : llm_ranking
  Strategy               : SCAN-AWARE

  ROUND PIPELINE (what happens each round)
  ------------------------------------------------------------------
  1. BASELINE INPUT
     Prompt: TASK 1: Sum

In [22]:
#@title Run optimization (feedback cycle)
campaign_rounds = await run_feedback_cycle_notebook(
    campaign_rounds, eval_data, campaign_config,
    svc=svc,
    pipeline_params=pipeline_params,
    scan_context=locals().get("scan_context"),
    experiment_id=locals().get("EXPERIMENT_ID"),
)

  Using stored baseline 20.0% (notebook had 10.0%)

╔════════════════════════════════════════════════════════════════════╗
║  FEEDBACK CYCLE STARTING                                           ║
╠════════════════════════════════════════════════════════════════════╣
║  Baseline       20.0%                                              ║
║  Max rounds     3              Patience    2                       ║
║  Candidates     5                                                  ║
║  Sample size    15 of 984                                          ║
║  Min detectable ±36.2% (α=0.05, 80% power)                         ║
║  Model          openai/gpt-oss-120b                                ║
║  L2 (refine)    disabled           L3 (plan)   disabled            ║
║  Scan context   YES                                                ║
║  Critique       enabled                                            ║
╚════════════════════════════════════════════════════════════════════╝


2026-03-18 16:29:03 INFO     [httpx] HTTP Request: POST http://127.0.0.1:8000/sessions "HTTP/1.1 200 OK"
2026-03-18 16:29:03 INFO     [api.services.campaign.feedback_cycle] Using provided baseline (acc=0.200)
2026-03-18 16:29:03 INFO     [api.services.campaign.feedback_cycle] Cycle identity: cycle_68e2c53845c3
2026-03-18 16:29:03 INFO     [api.services.campaign.feedback_cycle] Resuming cycle cycle_68e2c53845c3 — 2 prior round(s) on disk
2026-03-18 16:29:05 INFO     [api.services.obs.observability_logger] Dataset 'termnorm_ground_truth': 728 items registered, 256 duplicates/empty skipped (from 984 input)
2026-03-18 16:29:05 WARNING  [api.services.obs.observability_logger] Skipping Langfuse cloud dataset registration for 984 items (rate-limit risk). Use the dedicated Langfuse sync cell instead.
2026-03-18 16:29:05 INFO     [api.services.campaign.feedback_cycle] Registered 728 dataset items for 'termnorm_ground_truth'
2026-03-18 16:29:05 INFO     [api.services.campaign.feedback_cycle] Fee

  ✓ Initialized  cycle=cycle_68e2c5  samples=15  obs=ON
    Resumed from round 2 (2 rounds cached)

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  ROUND 1/3                                                 patience 0/2
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

├─ GENERATE ─────────────────────────────────────────────────────────────┤
│  Current best    20.0%
│  Prompt          You are a candidate evaluation expert.  Summari...
│  Candidates      5   Creativity: 0.7   Scan: YES   Critique: NO
│  Model           openai/gpt-oss-120b
│  Scan focus: 4 improving axes [max_token_candidates, query_prefix, profiling_schema, profiling_temperature]
│  Scan baseline: 10.0%
├────────────────────────────────────────────────────────────────────────┤
  ✓ 5 candidates generated (loaded from disk)
    C1: Increase max_token_candidates to 40 and set pro...
    C2: Swap query_prefix to a semantic variant and exp...
    C3: Raise raw_content_li

2026-03-18 16:29:05 INFO     [api.services.campaign.critique] Rich critique: 4388 chars prompt, round 1, acc=0.200



  ┌─ C5/5 ───────────────────────────────────── 6.7% [1.2%-29.8%] ─┐
  │  Explore a larger token budget (70) and hig...  pp=[max_token_candidates, profiling_temperature, query_prefix] +2│
  │  1/15 hits  composite=0.0600  vs baseline: -13.3%              │
  │  best so far: C4 20.0%                                         │
  └────────────────────────────────────────────────────────────────┘


...

  ✓ 5 candidates generated (from LLM)
    C1: Increase candidate pool size.
    C2: Use a more specific query prompt.
    C3: Add extra fields to the profiling schema.
    C4: Lower temperature and raise content limit.
    C5: Combine higher candidate count with a detailed ...

│  Settings diff (17 params, 7 SPs):
│                                             Start   Parent  C1      C2      C3      C4      C5      
│                       max_token_candidates  30      ·       35      30      ·       ·       40      
│                           profiling_schema  -       -       -       -       [a]     -       -       
│         profiling_schema.alternative_names  array   ·       ·       ·       -       array   ·       
│              profiling_schema.applications  [b]     ·       ·       ·       -       [b]     ·       
│    profiling_schema.classification_aliases  [c]     ·       ·       ·       -       [c]     ·       
│     profiling_schema.constituent_materials  [d]     ·       ·   

2026-03-18 16:30:05 WARNING  [api.services.prompt_eval] EscalationCheck 'degradation' triggered at query 5/15 (candidate 2/5)
2026-03-18 16:30:05 WARNING  [api.services.prompt_eval] Escalation 'degradation' aborted eval at 5/15 queries
2026-03-18 16:30:05 WARNING  [api.services.prompt_optimizer] Escalation 'degradation' — aborting remaining 3 candidates
2026-03-18 16:30:05 INFO     [api.services.campaign.critique] Rich critique: 4662 chars prompt, round 3, acc=0.200



  ┌─ C2/5 ─────────────────────────────────── 40.0% [11.8%-76.9%] ─┐
  │  Use a more specific query prompt.  pp=[query_prefix]          │
  │  2/5 hits  ⚠ aborted at 5/15  composite=0.3600  ⚠ 2/5 degraded  vs baseline: +20.0%│
  │  best so far: C2 40.0%                                         │
  └────────────────────────────────────────────────────────────────┘


2026-03-18 16:30:07 WARNING  [api.services.campaign.feedback_cycle] Escalation 'degradation' at round 2 — target=l2, degraded_rate=40.0%


  ┌─ SCOREBOARD ───────────────────────────────────────────────────────────────┐
  │  #   Label    Accuracy            95% CI  Composite    Delta               │
  │  1   C2         40.0%      [11.8%-76.9%]     0.3600    +20.0%  (aborted)   │
  │  2   C1          0.0%       [0.0%-20.4%]     0.0000    -20.0%  *           │
  └────────────────────────────────────────────────────────────────────────────┘
  ⚠ NO IMPROVEMENT  best candidate 20.0%
  Critique: Current settings cap candidate diversity, causing a plateau at 20% accuracy. Boost max_token_candidates and refine the query prefix to unlock higher‑quality candidates.

├─ ESCALATION ──────────────────────────────────────── degradation → l2 ─┤
│  Degraded: 40% of queries
│  web_search:partial_scrape: 2 occurrences
├────────────────────────────────────────────────────────────────────────┤
  Warning classifications:
    web_search:partial_scrape: under_investigation
  Patience reset to 0

╔════════════════════════════════════════════════

In [ ]:
#@title 5. Results — Campaign comparison, flip tracking, lineage
show_campaign_summary(campaign_rounds)
show_flip_tracking(campaign_rounds)
show_lineage_chain(campaign_rounds)

In [ ]:
#@title Save winner
save_campaign_winner(
    campaign_rounds, campaign_config, svc["store"], svc["backend_id"],
    experiment_id=locals().get("EXPERIMENT_ID"),
)

In [ ]:
#@title Generate LLM suggestions for next round
llm_client, llm_model = setup_llm(campaign_config)
suggestions = await generate_suggestions(
    campaign_rounds, eval_data, campaign_config,
    llm_client, model=llm_model,
)
display_suggestions(suggestions, len(campaign_rounds))
print("--- SUGGESTED CONFIG (copy to Setup) ---")
print(json.dumps(suggestions.get("suggested_config", campaign_config), indent=2))

In [ ]:
#@title Sync evaluation history to Langfuse
# Safe to re-run — already-pushed runs are skipped automatically.
stats = sync_langfuse(
    svc["store"], svc["backend_id"],
    dataset_name="termnorm_ground_truth",
)